### Import relevant packages

In [ ]:
import os
import time
import json
import base64
import requests
import pandas as pd
import lyricsgenius
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

### Pull APIs and Extract Relevant Data

In [53]:
load_dotenv(dotenv_path="/Users/stella/Documents/DSProj/setlist-predictor/.env")
API_KEY = os.getenv("SETLISTFM_API_KEY")
HEADERS = {
    "x-api-key": API_KEY,
    "Accept": "application/json"
}

response = requests.get(
    "https://api.setlist.fm/rest/1.0/search/artists",
    headers=HEADERS,
    params={"artistName": "Taylor Swift"}
)
print(response.status_code)
data = response.json()
data

200


{'type': 'artists',
 'itemsPerPage': 30,
 'page': 1,
 'total': 104,
 'artist': [{'mbid': 'f44a46fb-18ea-4c2c-b578-0826b6e67631',
   'name': '3LAU vs. Calvin Harris vs. Taylor Swift',
   'sortName': '3LAU vs. Harris, Calvin vs. Swift, Taylor',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/3lau-vs-calvin-harris-vs-taylor-swift-53c8e7a5.html'},
  {'mbid': '7288729a-b09e-4b5d-a9d6-a3ca8c42bd18',
   'name': 'Gracie Abrams feat. Taylor Swift',
   'sortName': 'Abrams, Gracie feat. Swift, Taylor',
   'disambiguation': '',
   'url': 'https://www.setlist.fm/setlists/gracie-abrams-feat-taylor-swift-4be51bde.html'},
  {'mbid': 'a26ca924-c7ba-4ab3-92e0-929145382fca',
   'name': 'Almost Eras: The Taylor Swift Experience',
   'sortName': 'Almost Eras: The Taylor Swift Experience',
   'disambiguation': 'Taylor Swift tribute band',
   'url': 'https://www.setlist.fm/setlists/almost-eras-the-taylor-swift-experience-2b9118ca.html'},
  {'mbid': '66fb2961-e3eb-42f4-ae64-640148d71cad',


In [55]:
def find_artist_mbid(artist_name, headers, exact_match=True):
    page = 1
    while True:
        response = requests.get(
            "https://api.setlist.fm/rest/1.0/search/artists",
            headers=headers,
            params={"artistName": artist_name, "p": page}
        )
        
        if response.status_code != 200:
            print(f"Stopped — status {response.status_code} on page {page}")
            print(response.json())
            break
        
        data = response.json()
        
        if "total" not in data:
            print(f"Unexpected response shape on page {page}:")
            print(data)
            break
        
        for artist in data.get("artist", []):
            if exact_match and artist["name"].lower() == artist_name.lower():
                return artist["mbid"], artist["name"]
            elif not exact_match and artist_name.lower() in artist["name"].lower():
                print(artist["name"], "-", artist["mbid"])
        
        total_pages = -(-data["total"] // data["itemsPerPage"])
        if page >= total_pages:
            break
        page += 1
        
        import time
        time.sleep(0.5)
    
    return None, None

In [56]:
mbid, name = find_artist_mbid("Taylor Swift", HEADERS)
print(mbid, name)

20244d07-534f-4eff-b4d4-930878889970 Taylor Swift


In [57]:
mbid = "20244d07-534f-4eff-b4d4-930878889970"
setlists = []
page = 1

while True:
    response = requests.get(
        f"https://api.setlist.fm/rest/1.0/artist/{mbid}/setlists",
        headers=HEADERS,
        params={"p": page}
    )
    if response.status_code != 200:
        print(f"Stopped at page {page}, status {response.status_code}")
        break
    
    page_data = response.json()
    setlists.extend(page_data.get("setlist", []))
    
    total_pages = -(-page_data["total"] // page_data["itemsPerPage"])
    print(f"Pulled page {page}/{total_pages}")
    
    if page >= total_pages:
        break
    page += 1
    
    time.sleep(1.5)

Pulled page 1/58
Pulled page 2/58
Pulled page 3/58
Pulled page 4/58
Pulled page 5/58
Pulled page 6/58
Pulled page 7/58
Pulled page 8/58
Pulled page 9/58
Pulled page 10/58
Pulled page 11/58
Pulled page 12/58
Pulled page 13/58
Pulled page 14/58
Pulled page 15/58
Pulled page 16/58
Pulled page 17/58
Pulled page 18/58
Pulled page 19/58
Pulled page 20/58
Pulled page 21/58
Pulled page 22/58
Pulled page 23/58
Pulled page 24/58
Pulled page 25/58
Pulled page 26/58
Pulled page 27/58
Pulled page 28/58
Pulled page 29/58
Pulled page 30/58
Pulled page 31/58
Pulled page 32/58
Pulled page 33/58
Pulled page 34/58
Pulled page 35/58
Pulled page 36/58
Pulled page 37/58
Pulled page 38/58
Pulled page 39/58
Pulled page 40/58
Pulled page 41/58
Pulled page 42/58
Pulled page 43/58
Pulled page 44/58
Pulled page 45/58
Pulled page 46/58
Pulled page 47/58
Pulled page 48/58
Pulled page 49/58
Pulled page 50/58
Pulled page 51/58
Pulled page 52/58
Pulled page 53/58
Pulled page 54/58
Pulled page 55/58
Pulled page 56/58
P

In [58]:
print(len(setlists)) 

1148


In [59]:
rows = []
for show in setlists:
    if not show["sets"]["set"]:
        continue
    
    for set_block in show["sets"]["set"]:
        set_name = set_block.get("name", "main")
        for song in set_block.get("song", []):
            rows.append({
                "event_date": show["eventDate"],
                "venue": show["venue"]["name"],
                "city": show["venue"]["city"]["name"],
                "country": show["venue"]["city"]["country"]["name"],
                "tour": show.get("tour", {}).get("name", None),
                "set_name": set_name,
                "song_name": song.get("name"),
                "is_cover": "cover" in song,
                "info": song.get("info", None)
            })

df = pd.DataFrame(rows)
df["event_date"] = pd.to_datetime(df["event_date"], format="%d-%m-%Y")
df.head()

,event_date,venue,city,country,tour,set_name,song_name,is_cover,info
0,2026-06-09,Dolby Theatre,Los Angeles,United States,NaN,main,"I Knew It, I Knew You",False,live debut; on piano
1,2026-06-09,Dolby Theatre,Los Angeles,United States,NaN,main,You've Got a Friend in Me,True,Taylor introduced Randy Newman
2,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,main,,False,"w/ elements of MA&tHP, The Alchemy, Fearless, ..."
3,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Miss Americana & the Heartbreak Prince,False,shortened
4,2024-12-08,BC Place Stadium,Vancouver,Canada,The Eras Tour,Lover,Cruel Summer,False,extended outro


In [60]:
df.to_csv("../data/taylor_swift_setlists.csv", index=False)

In [ ]:
all_songs = df["song_name"].unique()
print(len(all_songs))

## Spotify Tracklists Data

In [85]:
SPOTIFY_CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
SPOTIFY_CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

auth_string = f"{SPOTIFY_CLIENT_ID}:{SPOTIFY_CLIENT_SECRET}"
auth_bytes = base64.b64encode(auth_string.encode()).decode()

token_response = requests.post(
    "https://accounts.spotify.com/api/token",
    headers={"Authorization": f"Basic {auth_bytes}"},
    data={"grant_type": "client_credentials"}
)
spotify_token = token_response.json()["access_token"]

In [82]:
def get_album_tracklist(album_name, artist_name, token, before_year=None):
    headers = {"Authorization": f"Bearer {token}"}
    search_resp = requests.get(
        "https://api.spotify.com/v1/search",
        headers=headers,
        params={"q": f"album:{album_name} artist:{artist_name}", "type": "album", "limit": 10}
    )
    results = search_resp.json()["albums"]["items"]
    
    # exact (case-insensitive) name match only — excludes "Surprise Song Playlist", "(Special)", etc.
    original_candidates = [r for r in results if r["name"].strip().lower() == album_name.strip().lower()]
    
    if before_year:
        original_candidates = [r for r in original_candidates if int(r["release_date"][:4]) <= before_year]
    
    if not original_candidates:
        print(f"No exact match found for {album_name}, showing all candidates:")
        for r in results:
            print(" -", r["name"], "|", r["album_type"], "|", r["release_date"])
        return []
    
    original_candidates.sort(key=lambda r: r["release_date"])
    chosen = original_candidates[0]
    print(f"Matched: {chosen['name']} ({chosen['release_date']}, type={chosen['album_type']})")
    
    tracks_resp = requests.get(
        f"https://api.spotify.com/v1/albums/{chosen['id']}/tracks",
        headers=headers,
        params={"limit": 50}
    )
    return [t["name"] for t in tracks_resp.json()["items"]]

In [83]:
album_name_overrides = {
    "Fearless": "Fearless (Platinum Edition)"
}

albums = [
    "Fearless", "Speak Now", "Red", "1989", "reputation",
    "Lover", "folklore", "evermore", "Midnights", "THE TORTURED POETS DEPARTMENT",
    "The Life of a Showgirl"
]

rows = []
for album_name in albums:
    search_name = album_name_overrides.get(album_name, album_name)
    tracks = get_album_tracklist(search_name, "Taylor Swift", spotify_token)
    for track in tracks:
        rows.append({"album": album_name, "track_name": track}) 

tracklists_df = pd.DataFrame(rows)
print(tracklists_df.shape)
tracklists_df.head(20)

Matched: Fearless (Platinum Edition) (2008-11-11, type=album)
Matched: Speak Now (2010-10-25, type=album)
Matched: Red (2012-10-22, type=album)
Matched: 1989 (2014-01-01, type=album)
Matched: reputation (2017-11-10, type=album)
Matched: Lover (2019-08-23, type=album)
Matched: folklore (2020-07-24, type=album)
Matched: evermore (2020-12-10, type=album)
Matched: Midnights (2022-10-21, type=album)
Matched: THE TORTURED POETS DEPARTMENT (2024-04-18, type=album)
Matched: The Life of a Showgirl (2025-10-03, type=album)
(167, 2)


,album,track_name
0,Fearless,Jump Then Fall
1,Fearless,Untouchable
2,Fearless,Forever & Always - Piano Version
3,Fearless,Come In With The Rain
4,Fearless,SuperStar
5,Fearless,The Other Side Of The Door
6,Fearless,Fearless
7,Fearless,Fifteen
8,Fearless,Love Story
9,Fearless,Hey Stephen


In [84]:
tracklists_df.to_csv("../data/taylor_swift_album_tracklists.csv", index=False)

In [ ]:
album_songs_lookup = tracklists_df.groupby("album")["track_name"].apply(set).to_dict()
showgirl_songs = album_songs_lookup.get("The Life of a Showgirl", set())
print(len(showgirl_songs))

In [69]:
def get_album_tracklist_debug(album_name, artist_name, token):
    headers = {"Authorization": f"Bearer {token}"}
    search_resp = requests.get(
        "https://api.spotify.com/v1/search",
        headers=headers,
        params={"q": f"album:{album_name} artist:{artist_name}", "type": "album", "limit": 10}
    )
    results = search_resp.json()["albums"]["items"]
    for r in results:
        print(r["name"], "-", r["release_date"], "-", r["id"])

get_album_tracklist_debug("reputation", "Taylor Swift", spotify_token)

reputation - 2017-11-10 - 6DEjYFkNZh67HP7R9PSZvv
reputation (Big Machine Radio Release Special) - 2017-11-10 - 1Hrs3jLGexOvBoaPMoOQYJ
reputation Stadium Tour Surprise Song Playlist - 2017-11-09 - 1MPAXuTVL2Ej5x0JHiSPq8
Taylor Swift Karaoke: reputation - 2018-03-09 - 1MHuZZrGT36cXLxAQ5cLP3


## Genius Lyrics Data

In [ ]:
load_dotenv(dotenv_path="../.env")
genius = lyricsgenius.Genius(os.getenv("GENIUS_ACCESS_TOKEN"), timeout=15, retries=3)
genius.verbose = False

# resume from previous pulls if the file already exists, otherwise start fresh
try:
    with open("../data/taylor_swift_lyrics.json") as f:
        lyrics_data = json.load(f)
except FileNotFoundError:
    lyrics_data = {}

# all_songs should include both the historical catalog and the Showgirl tracklist
songs_to_pull = set(all_songs) | set(showgirl_songs)

for song in songs_to_pull:
    if song in lyrics_data:
        continue
    try:
        song_result = genius.search_song(song, "Taylor Swift")
        lyrics_data[song] = song_result.lyrics if song_result else None
        print(f"OK: {song}")
    except Exception as e:
        print(f"Failed on {song}: {e}")
        lyrics_data[song] = None
    time.sleep(0.5)

with open("../data/taylor_swift_lyrics.json", "w") as f:
    json.dump(lyrics_data, f)

print(f"Pulled lyrics for {sum(v is not None for v in lyrics_data.values())}/{len(lyrics_data)} songs")

## Last.fm Playcount Data

In [ ]:
load_dotenv(dotenv_path="../.env")
LASTFM_API_KEY = os.getenv("LASTFM_API_KEY")

def get_lastfm_playcount(track_name, artist_name="Taylor Swift"):
    response = requests.get(
        "http://ws.audioscrobbler.com/2.0/",
        params={
            "method": "track.getInfo",
            "api_key": LASTFM_API_KEY,
            "artist": artist_name,
            "track": track_name,
            "format": "json"
        }
    )
    data = response.json()
    if "track" in data:
        return int(data["track"].get("playcount", 0))
    return None

playcount_scores = {}
for song in set(all_songs) | set(showgirl_songs)::  # or all_songs, whichever list covers your full catalog + Showgirl
    if song in playcount_scores:
        continue
    try:
        pc = get_lastfm_playcount(song)
        playcount_scores[song] = pc
        print(f"{song}: {pc}")
    except Exception as e:
        print(f"Failed on {song}: {e}")
        playcount_scores[song] = None
    time.sleep(0.3)

with open("../data/taylor_swift_playcounts.json", "w") as f:
    json.dump(playcount_scores, f)